# 手语识别模型 - 结果分析与评估

---

## 1. 加载结果数据

In [ ]:
import json
import os
LOG_DIR = 'log'
CHECKPOINT_DIR = 'checkpoints'

with open(os.path.join(LOG_DIR, 'results.json'), 'r') as f:
    results = json.load(f)

with open(os.path.join(LOG_DIR, 'evaluation_results.json'), 'r') as f:
    eval_results = json.load(f)

print(f"最佳模型: {eval_results['model']}")
print(f"测试准确率: {eval_results['test_accuracy']:.4f}")

## 2. 模型性能比较

In [ ]:
import pandas as pd

comparison_data = []
for name, res in results.items():
    comparison_data.append({
        '模型': name,
        '最佳验证准确率': res['best_val_acc'],
        '训练时间(秒)': res['train_time']
    })

df = pd.DataFrame(comparison_data)
df = df.sort_values('最佳验证准确率', ascending=False)
df

## 3. 训练曲线可视化

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(LOG_DIR, 'plots', 'training_curves.png')))

## 4. 混淆矩阵

In [ ]:
display(Image(filename=os.path.join(LOG_DIR, 'plots', 'confusion_matrix.png')))

## 5. 分类报告

In [ ]:
report = eval_results['classification_report']

metrics_data = []
for label, metrics in report.items():
    if label not in ['accuracy', 'macro avg', 'weighted avg']:
        metrics_data.append({
            '类别': label,
            '精确率': metrics['precision'],
            '召回率': metrics['recall'],
            'F1分数': metrics['f1-score'],
            '样本数': metrics['support']
        })

df_metrics = pd.DataFrame(metrics_data)
df_metrics = df_metrics.sort_values('F1分数', ascending=True)
df_metrics

## 6. 样本预测结果

In [ ]:
display(Image(filename=os.path.join(LOG_DIR, 'plots', 'sample_predictions.png')))

## 7. 结果分析

In [ ]:
print("=" * 60)
print("模型性能分析")
print("=" * 60)

best_model = eval_results['model']
test_acc = eval_results['test_accuracy']
best_val_acc = results[best_model]['best_val_acc']

print(f"最佳模型: {best_model}")
print(f"验证准确率: {best_val_acc:.4f}")
print(f"测试准确率: {test_acc:.4f}")
print(f"泛化差距: {abs(best_val_acc - test_acc):.4f}")

report = eval_results['classification_report']
worst_class = min([(k, v['f1-score']) for k, v in report.items() 
                  if k not in ['accuracy', 'macro avg', 'weighted avg']], 
                 key=lambda x: x[1])[0]
print(f"\nF1最低的类别: {worst_class}")
print(f"F1分数: {report[worst_class]['f1-score']:.4f}")

---

# 8. 超参数实验结果

## 8.1 加载超参数实验数据

In [ ]:
HYPERPARAM_DIR = 'log/hyperparam'

with open(os.path.join(HYPERPARAM_DIR, 'summary.json'), 'r') as f:
    hyperparam_summary = json.load(f)

print("超参数实验汇总加载完成")
print(f"推荐配置: {hyperparam_summary['recommended']}")

## 8.2 超参数实验结果表格

In [ ]:
import pandas as pd

exp_data = []

param_configs = {
    'batch_size': ['16', '32', '64', '128', '256'],
    'epochs': ['5', '10', '20', '30'],
    'learning_rate': ['0.0001', '0.0005', '0.001', '0.005', '0.01'],
    'optimizer': ['Adam', 'SGD', 'RMSprop'],
    'activation': ['ReLU', 'GELU', 'ELU']
}

for param, configs in param_configs.items():
    for cfg in configs:
        exp_name = f"{param}_{cfg}"
        result = hyperparam_summary.get(exp_name, {})
        exp_data.append({
            '超参数': param,
            '参数值': cfg,
            '验证准确率': f"{result.get('best_val_acc', 0)*100:.2f}%",
            '训练时间(秒)': f"{result.get('train_time', 0):.2f}"
        })

df_hyperparam = pd.DataFrame(exp_data)
df_hyperparam

## 8.3 超参数对比可视化

In [ ]:
display(Image(filename=os.path.join(HYPERPARAM_DIR, 'plots', 'hyperparam_comparison.png')))

## 8.4 训练时间对比

In [ ]:
display(Image(filename=os.path.join(HYPERPARAM_DIR, 'plots', 'time_comparison.png')))

## 8.5 学习曲线对比

In [ ]:
display(Image(filename=os.path.join(HYPERPARAM_DIR, 'plots', 'training_curves.png')))

## 8.6 损失曲线对比

In [ ]:
display(Image(filename=os.path.join(HYPERPARAM_DIR, 'plots', 'loss_curves.png')))

## 8.7 超参数实验分析结论

In [ ]:
print("=" * 60)
print("超参数实验分析结论")
print("=" * 60)

print("\n1. 模型收敛迅速: EnhancedCNN在各超参数配置下均能快速收敛")
print("\n2. 超参数敏感性: 验证准确率对大多数超参数变化不敏感")
print("\n3. 训练效率: batch_size越大训练时间越短")
print("\n4. 最佳实践: 建议使用默认配置batch_size=128, epochs=20")

print("\n各超参数最佳值:")
for param in ['batch_size', 'epochs', 'learning_rate', 'optimizer', 'activation']:
    best = hyperparam_summary.get(param, {})
    print(f"  - {param}: {best.get('best_value')} (准确率: {best.get('best_acc')*100:.2f}%)")

---

# 9. 反思与改进

### 9.1 ���型局限性
- 简单CNN准确率较低，模型容量不足
- 迁移学习模型针对ImageNet设计，可能不完全适配灰度手语图像
- 预训练模型首次下载时间较长

### 9.2 数据集局限性
- 28x28分辨率较低
- 部分手势视觉相似（如D与E）
- 类内变化可能较大

### 9.3 改进方案
- 数据增强（旋转、平移、缩放）
- 更先进的网络架构（EfficientNet）
- 超参数调优（学习率调度）
- 集成学习

### 9.4 参考文献
1. K. He et al., "Deep Residual Learning for Image Recognition," CVPR 2016.
2. S. Ioffe and C. Szegedy, "Batch Normalization," ICML 2015.